In [ ]:
!pip install proglearn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import nibabel as nb
from scipy import ndimage
import scipy.stats as ss
from joblib import Parallel, delayed

In [ ]:
df = pd.read_excel('/content/Human.parcellated_thickness.xlsx')

In [ ]:
df.head()

,Unnamed: 0,sid,Markov.1,Markov.2,Markov.3,Markov.4,Markov.5,Markov.6,Markov.7,Markov.8,...,Schaefer217.191,Schaefer217.192,Schaefer217.193,Schaefer217.194,Schaefer217.195,Schaefer217.196,Schaefer217.197,Schaefer217.198,Schaefer217.199,Schaefer217.200
0,0,sub-OAS30876MRD4592,1.995032,2.203564,1.651978,1.969754,2.603206,2.295727,2.385144,2.719692,...,8.193966,7.736098,7.404804,7.431338,7.541022,7.433447,7.475594,7.460332,7.476401,7.609443
1,1,sub-HBN_CBIC_NDARXC962XNK,2.557198,2.011555,2.175673,1.863080,2.473705,2.576267,2.392282,2.242582,...,6.927265,7.487809,7.098700,6.753360,6.841211,6.707229,7.189156,6.795299,6.823550,6.533783
2,2,sub-AOMIC_0770,2.246607,2.295872,1.978412,2.069700,2.213602,2.449572,2.541624,2.777280,...,8.117053,7.775745,7.608771,7.579930,7.573511,7.607256,7.883723,7.630075,7.670835,7.354955
3,3,sub-AOMIC_0344,2.219745,2.366237,2.036068,2.173696,2.508847,2.408997,2.430510,2.882698,...,7.908504,7.856069,7.756918,7.526684,7.410575,7.654072,7.952062,7.682724,7.444550,7.697996
4,4,sub-Narratives_150,2.131236,2.432549,2.066158,2.352589,2.331565,2.799966,2.490590,2.818574,...,8.229232,7.847060,7.821367,7.739102,8.001608,7.803380,7.750091,7.730715,7.858832,7.892026


In [ ]:
df_sex = pd.read_excel('/content/subjects_age_sex_data_MRI.xlsx')
df_sex.head()

,ID,Age,Sex,Dataset,Dataset-ID
0,sub-ABIDE1050339,18.0000,MALE,ABIDE,50339
1,sub-ABIDE1050701,18.0000,MALE,ABIDE,50701
2,sub-ABIDE1050445,18.1383,MALE,ABIDE,50445
3,sub-ABIDE1050459,18.1547,MALE,ABIDE,50459
4,sub-ABIDE1050341,18.2000,FEMALE,ABIDE,50341


In [ ]:
X1 = []
X2 = []
y_human = []
IDs = set(df['sid'])
ref_IDs = set(df_sex['ID'])

for subject in tqdm(IDs):
    if subject in ref_IDs:
        features = np.array(df[df['sid']==subject]).reshape(-1)[2:]
        gender = list(df_sex[df_sex['ID']==subject]['Sex'])
        sex = int(gender[0]=='FEMALE')

        X1.append(list(features[:182]))
        X2.append(list(features[182:]))
        y_human.append(sex)

X1_human = np.array(X1)
X2_human = np.array(X2)

100%|██████████| 14465/14465 [00:42<00:00, 340.76it/s]


In [ ]:
print(X1_human.shape, X2_human.shape, len(y_human))

(10648, 182) (10648, 200) 10648


In [ ]:
df = pd.read_excel('/content/Macaque.parcellated_thickness.xlsx')
df.head()

,Unnamed: 0,participant_id,age,sex,Markov.1,Markov.2,Markov.3,Markov.4,Markov.5,Markov.6,...,Schaefer217.191,Schaefer217.192,Schaefer217.193,Schaefer217.194,Schaefer217.195,Schaefer217.196,Schaefer217.197,Schaefer217.198,Schaefer217.199,Schaefer217.200
0,0,sub-1001,1.756164,M,3.048436,3.908286,3.221595,3.615675,4.662432,3.707754,...,4.231826,4.908868,4.522730,2.294943,2.853976,3.406234,4.261370,4.131977,3.387978,3.451267
1,1,sub-1002,1.783562,F,3.053520,3.748308,3.043567,3.764927,4.708283,4.060617,...,4.384853,4.849508,4.589500,2.443734,2.855187,3.344378,3.926697,3.477919,2.962553,3.474969
2,2,sub-1003,1.756164,M,3.211265,4.122524,3.374628,4.022762,4.759439,4.182558,...,4.570739,4.921833,4.770724,3.106145,3.094785,3.350355,4.562199,4.212585,3.582792,3.827813
3,3,sub-1004,1.756164,M,3.004275,3.681716,3.227427,3.762712,4.555942,3.984013,...,4.264869,4.935628,4.505048,3.337418,2.892611,3.690076,4.095378,4.328465,3.763171,3.758017
4,4,sub-1005,1.742466,M,2.868796,3.837011,2.997172,3.724171,4.537298,3.816082,...,4.154663,4.817727,4.695378,3.965287,3.219764,3.268439,4.115168,3.889531,3.271547,4.040183


In [ ]:
df_sex = pd.read_csv('/content/uwmadison.csv')
df_sex.head()

,participant_id,age,sex
0,sub-1001,1.756164,M
1,sub-1002,1.783562,F
2,sub-1003,1.756164,M
3,sub-1004,1.756164,M
4,sub-1005,1.742466,M


In [ ]:
X1 = []
X2 = []
y_monkey = []
IDs = set(df['participant_id'])
ref_IDs = set(df_sex['participant_id'])

for subject in tqdm(IDs):
    if subject in ref_IDs:
        features = np.array(df[df['participant_id']==subject]).reshape(-1)[4:]
        gender = list(df_sex[df_sex['participant_id']==subject]['sex'])
        sex = int(gender[0]=='F')

        X1.append(list(features[:182]))
        X2.append(list(features[182:]))
        y_monkey.append(sex)

X1_monkey = np.array(X1)
X2_monkey = np.array(X2)

100%|██████████| 592/592 [00:00<00:00, 928.59it/s]


In [ ]:
print(X1_monkey.shape, X2_monkey.shape, len(y_monkey))

(592, 182) (592, 200) 592


## Try random forest


data preparation

In [ ]:
X1_human = np.nan_to_num(X1_human)
X2_human = np.nan_to_num(X2_human)
X_human = np.concatenate((X1_human,X2_human),axis=1)

X1_monkey = np.nan_to_num(X1_monkey)
X2_monkey = np.nan_to_num(X2_monkey)
X_monkey = np.concatenate((X1_monkey, X2_monkey),axis=1)

X1_monkey_train, X1_monkey_test, y_monkey_train, y_monkey_test = train_test_split(X1_monkey, y_monkey, train_size=0.8, random_state=42, stratify=y_monkey)
X2_monkey_train, X2_monkey_test, y_monkey_train, y_monkey_test = train_test_split(X2_monkey, y_monkey, train_size=0.8, random_state=42, stratify=y_monkey)
X_monkey_train = np.concatenate((X1_monkey_train, X2_monkey_train),axis=1)
X_monkey_test = np.concatenate((X1_monkey_test, X2_monkey_test),axis=1)

X1_combined = np.concatenate((X1_human,X1_monkey_train),axis=0)
X2_combined = np.concatenate((X2_human,X2_monkey_train),axis=0)
X_combined = np.concatenate((X1_combined,X2_combined),axis=1)
y_combined = np.concatenate((y_human,y_monkey_train),axis=0)

# Markov

1. Train on human, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X1_human, y_human, train_size=0.8, random_state=ii, stratify=y_human)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train, y_train)
    accuracy += np.mean(clf.predict(X1_monkey)==y_monkey)

print('Accuracy is ',accuracy/reps)

(11121, 382) 119


100%|██████████| 5/5 [00:23<00:00,  4.60s/it]

Accuracy is  0.555954169797145


2. Train on human and monkey, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X1_combined, y_combined, train_size=0.8, random_state=ii, stratify=y_combined)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train, y_train)
    accuracy += np.mean(clf.predict(X1_monkey_test)==y_monkey_test)

print('Accuracy is ',accuracy/reps)

3. Train on human then on monkey, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X1_human, y_human, train_size=0.8, random_state=ii, stratify=y_human)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train, y_train)
    clf.fit(X1_monkey_train, y_monkey_train)
    accuracy += np.mean(clf.predict(X1_monkey_test)==y_monkey_test)

print('Accuracy is ',accuracy/reps)

# Schaefer

1. Train on human, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X2_human, y_human, train_size=0.8, random_state=ii, stratify=y_human)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train,y_train)
    accuracy += np.mean(clf.predict(X2_monkey)==y_monkey)

print('Accuracy is ',accuracy/reps)

100%|██████████| 5/5 [01:25<00:00, 17.10s/it]

Accuracy is  0.4991596638655462


2. Train on human and monkey, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X2_combined, y_combined, train_size=0.8, random_state=ii, stratify=y_combined)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train, y_train)
    accuracy += np.mean(clf.predict(X2_monkey_test)==y_monkey_test)

print('Accuracy is ',accuracy/reps)

3. Train on human then monkey, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X2_human, y_human, train_size=0.8, random_state=ii, stratify=y_human)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train, y_train)
    clf.fit(X2_monkey_train, y_monkey_train)
    accuracy += np.mean(clf.predict(X2_monkey_test)==y_monkey_test)

print('Accuracy is ',accuracy/reps)

# Markov+Schaefer

1. Train on human, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X_human, y_human, train_size=0.8, random_state=ii, stratify=y_human)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train,y_train)
    accuracy += np.mean(clf.predict(X_monkey)==y_monkey)

print('Accuracy is ',accuracy/reps)

2. Train on human and monkey, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X_combined, y_combined, train_size=0.8, random_state=ii, stratify=y_combined)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train, y_train)
    accuracy += np.mean(clf.predict(X_monkey_test)==y_monkey_test)

print('Accuracy is ',accuracy/reps)

3. Train on human then on monkey, test on monkey

In [ ]:
reps = 5
accuracy = 0.0

for ii in tqdm(range(reps)):
    x_train, x_test, y_train, y_test = train_test_split(
                    X_human, y_human, train_size=0.8, random_state=ii, stratify=y_human)
    clf = RandomForestClassifier(n_estimators=1000, n_jobs=-1)
    clf.fit(x_train, y_train)
    clf.fit(X_monkey_train, y_monkey_train)
    accuracy += np.mean(clf.predict(X_monkey_test)==y_monkey_test)

print('Accuracy is ',accuracy/reps)